KPIs de marketing

In [21]:
import pandas as pd
import numpy as np
import os
import random
import matplotlib.pyplot as plt

RUTA BASE FIJA

In [15]:
BASE_DIR = r"C:\Users\Yudith\Desktop\Análisis de Datos\AUTOMATIZACIÓN PYTHON NUBE"

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

RAW_PATH = os.path.join(RAW_DIR, "campañas_marketing.csv")
CLEAN_PATH = os.path.join(PROCESSED_DIR, "campañas_limpio.csv")
KPIS_PATH = os.path.join(PROCESSED_DIR, "kpis_por_canal.csv")

CREAR DATASET 

In [16]:
if not os.path.exists(RAW_PATH):
    print("⚠️ No existe dataset crudo → generando uno ficticio...")

    n = 300
    canales = ["Google Ads", "Meta Ads", "TikTok Ads", "Email", "Organic"]
    campañas = ["Promo Verano", "Black Friday", "Lanzamiento App", "Retargeting"]

    data = {
        "channel": [random.choice(canales) for _ in range(n)],
        "campaign": [random.choice(campañas) for _ in range(n)],
        "impressions": np.random.randint(1000, 50000, n),
        "clicks": np.random.randint(50, 5000, n),
        "conversions": np.random.randint(5, 500, n),
        "cost": np.round(np.random.uniform(50, 5000, n), 2),
        "revenue": np.round(np.random.uniform(100, 12000, n), 2),
    }

    pd.DataFrame(data).to_csv(RAW_PATH, index=False)
    print("✅ Dataset ficticio creado en:")
    print(RAW_PATH)

 LEER DATASET

In [17]:
df = pd.read_csv(RAW_PATH)
print("\n📥 Dataset cargado:", RAW_PATH)
print("🧾 Columnas:", list(df.columns))



📥 Dataset cargado: C:\Users\Yudith\Desktop\Análisis de Datos\AUTOMATIZACIÓN PYTHON NUBE\data\raw\campañas_marketing.csv
🧾 Columnas: ['channel', 'campaign', 'impressions', 'clicks', 'conversions', 'cost', 'revenue']


LIMPIEZA

In [19]:
df["ctr"] = np.where(df["impressions"] > 0, df["clicks"] / df["impressions"], 0)
df["cpc"] = np.where(df["clicks"] > 0, df["cost"] / df["clicks"], 0)
df["cpa"] = np.where(df["conversions"] > 0, df["cost"] / df["conversions"], 0)
df["roi"] = np.where(df["cost"] > 0, (df["revenue"] - df["cost"]) / df["cost"], 0)

df.to_csv(CLEAN_PATH, index=False)
print("\n✅ Dataset limpio guardado en:")
print(CLEAN_PATH)



✅ Dataset limpio guardado en:
C:\Users\Yudith\Desktop\Análisis de Datos\AUTOMATIZACIÓN PYTHON NUBE\data\processed\campañas_limpio.csv


In [ ]:
 KPIs FILA A FILA

In [18]:
df = df.drop_duplicates()
df = df.replace([np.inf, -np.inf], np.nan)

numeric_cols = ["impressions", "clicks", "conversions", "cost", "revenue"]

df[numeric_cols] = (
    df[numeric_cols]
      .apply(pd.to_numeric, errors="coerce")
      .fillna(0)
      .clip(lower=0)
)

 KPIs POR CANAL

In [20]:
kpis_canal = (
    df.groupby("channel", as_index=False)
      .agg(
          impressions=("impressions", "sum"),
          clicks=("clicks", "sum"),
          conversions=("conversions", "sum"),
          cost=("cost", "sum"),
          revenue=("revenue", "sum")
      )
)

kpis_canal["ctr"] = np.where(kpis_canal["impressions"] > 0, kpis_canal["clicks"] / kpis_canal["impressions"], 0)
kpis_canal["cpc"] = np.where(kpis_canal["clicks"] > 0, kpis_canal["cost"] / kpis_canal["clicks"], 0)
kpis_canal["cpa"] = np.where(kpis_canal["conversions"] > 0, kpis_canal["cost"] / kpis_canal["conversions"], 0)
kpis_canal["roi"] = np.where(kpis_canal["cost"] > 0, (kpis_canal["revenue"] - kpis_canal["cost"]) / kpis_canal["cost"], 0)

kpis_canal.to_csv(KPIS_PATH, index=False)

print("\n✅ KPIs por canal generados:")
print(KPIS_PATH)
print("📊 Canales:", len(kpis_canal))


✅ KPIs por canal generados:
C:\Users\Yudith\Desktop\Análisis de Datos\AUTOMATIZACIÓN PYTHON NUBE\data\processed\kpis_por_canal.csv
📊 Canales: 5


GENERAR GRÁFICOS

CTR POR CANAL

In [24]:
plt.figure()
plt.bar(df["channel"], df["ctr"])
plt.title("CTR por Canal")
plt.ylabel("CTR")
plt.xlabel("Canal")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ctr_por_canal.png"))
plt.close()

CPA POR CANAL

In [26]:
plt.figure()
plt.bar(df["channel"], df["cpa"])
plt.title("CPA por Canal")
plt.ylabel("Costo por Adquisición")
plt.xlabel("Canal")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cpa_por_canal.png"))
plt.close()

ROI POR CANAL

✅ Gráficos generados correctamente
📁 Carpeta: C:\Users\Yudith\Desktop\Análisis de Datos\AUTOMATIZACIÓN PYTHON NUBE\reports


EXPORTAR REORTE

In [13]:
import pandas as pd
import os

# =====================================================
# BASE DIR
# =====================================================
BASE_DIR = os.getcwd()

INPUT_PATH = os.path.join(
    BASE_DIR, "data", "processed", "kpis_por_canal.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR, "reports", "reporte_kpis_marketing.xlsx"
)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# =====================================================
# LECTURA
# =====================================================
df = pd.read_csv(INPUT_PATH)

# =====================================================
# EXPORTAR REPORTE
# =====================================================
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="KPIs por Canal", index=False)

print("✅ REPORTE FINAL GENERADO")
print("📊 KPIs incluidos")
print("📁 Archivo:", OUTPUT_PATH)


✅ REPORTE FINAL GENERADO
📊 KPIs incluidos
📁 Archivo: C:\Users\Yudith\Desktop\Análisis de Datos\AUTOMATIZACIÓN PYTHON NUBE\reports\reporte_kpis_marketing.xlsx
